# Clip a Texas FEMA eBFE model to one NWM reach

This notebook creates a real 1D breakout from the FEMA Base Level Engineering
(eBFE) model for **Shiloh Branch** in the Lower Colorado–Cummins study
(HUC8 12090301). The target is one actual National Water Model v3 flowline:
**NWM feature 5790868**.

The workflow is deliberately extent-first:

1. Load the HEC-RAS model footprint and geometry.
2. Query NOAA's public
   [static NWM flowline service](https://maps.water.noaa.gov/server/rest/services/reference/static_nwm_flowlines/FeatureServer/0).
3. Measure how much of every NWM edge lies inside the model polygon—no weighted
   seven-signal match is needed.
4. Select the target edge's intersected cross sections and add the standard one
   cross-section downstream overlap.
5. Write, validate, and run an independent `RasBreakout1D` project.
6. Compare retained geometry and hydraulic results with the source model.

The NWM subset is cached as **GeoParquet** in the run workspace. The source eBFE
workspace defaults to `H:\Testing\eBFE Model Organization`; set
`RAS_COMMANDER_EBFE_ROOT` to use another organized cache.


In [ ]:
from datetime import datetime
from pathlib import Path
import io
import os
import shutil
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "ras_commander").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import ras_commander
from ras_commander import (
    GeomParser,
    HdfProject,
    HdfXsec,
    RasBreakout1D,
    RasCmdr,
    RasNetworkConflation,
    RasPrj,
    init_ras_project,
)

print(f"ras-commander: {ras_commander.__version__}")
print(f"Loaded from: {ras_commander.__file__}")
WORK_ROOT = REPO_ROOT / "working" / "235_1d_breakout_model"
RUN_ROOT = WORK_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
EBFE_WORKSPACE = Path(
    os.environ.get("RAS_COMMANDER_EBFE_ROOT", r"H:\Testing\eBFE Model Organization")
)
SOURCE_MODEL_DIR = (
    EBFE_WORKSPACE
    / "Organized"
    / "LowerColoradoCummins_12090301"
    / "RAS Model"
    / "Rabbs Creek-Colorado River"
    / "SHILOH BRANCH"
)
SOURCE_PROJECT_NAME = "SHILOH BRANCH.prj"
PROJECT_CRS = "EPSG:2277"  # NAD83 / Texas Central (ftUS), recorded in SHILOH BRANCH.xml
RAS_VERSION = "7.0"
PLAN_NUMBER = "01"
RIVER = "SHILOH BRANCH"
REACH = "Reach-1"
TARGET_EDGE_ID = "5790868"

if not (SOURCE_MODEL_DIR / SOURCE_PROJECT_NAME).is_file():
    raise FileNotFoundError(
        "The organized Lower Colorado–Cummins eBFE model was not found. "
        "Set RAS_COMMANDER_EBFE_ROOT to the eBFE Model Organization workspace."
    )
if str(RUN_ROOT).startswith("\\"):
    raise RuntimeError("Use a local or mapped-drive working path for HEC-RAS execution.")

RUN_ROOT.mkdir(parents=True, exist_ok=False)
source_copy = RUN_ROOT / "source"
shutil.copytree(SOURCE_MODEL_DIR, source_copy)
source_project = source_copy / SOURCE_PROJECT_NAME

source_ras = RasPrj()
init_ras_project(
    source_project,
    RAS_VERSION,
    ras_object=source_ras,
    hide_intro=True,
)
source_plan = source_ras.plan_df.loc[
    source_ras.plan_df["plan_number"].astype(str).str.zfill(2) == PLAN_NUMBER
].iloc[0]
source_geometry = Path(source_plan["Geom Path"])
source_geometry_hdf = Path(f"{source_geometry}.hdf")

display(
    source_ras.plan_df[[
        "plan_number", "Plan Title", "flow_type", "geometry_type",
        "num_cross_sections", "Geom Path", "Flow Path",
    ]]
)
print(f"Run workspace: {RUN_ROOT}")


## 1. Recompute the source model and load its actual footprint

The delivered model is a one-reach, 40-cross-section steady model. Its legacy
eBFE XML file records the Texas Central feet CRS even though the RAS 4.1-era HDF
does not embed it. We assign that authoritative CRS after reading the geometry.
The source is recomputed with HEC-RAS 7.0 so the later comparison uses fresh,
successful results from the copied workspace.


In [ ]:
source_compute = RasCmdr.compute_plan(
    PLAN_NUMBER,
    ras_object=source_ras,
    clear_geompre=True,
    force_rerun=True,
    num_cores=1,
    verify=True,
)
assert source_compute, "Source steady plan did not complete successfully"

source_plan_hdf = Path(f"{source_plan['full_path']}.hdf")
assert source_plan_hdf.is_file()

model_extent, model_bounds = HdfProject.get_project_extent(
    source_geometry_hdf,
    include_1d=True,
    include_2d=False,
    include_storage=False,
    buffer_percent=0,
    geometry_type="footprint",
)
model_extent = model_extent.set_crs(PROJECT_CRS, allow_override=True)
model_extent["geometry_id"] = "shiloh-g01"

source_xs = GeomParser.get_xs_cut_lines(source_geometry).set_crs(
    PROJECT_CRS, allow_override=True
)
source_xs["station_num"] = pd.to_numeric(source_xs["station"], errors="coerce")
source_centerline = GeomParser.get_river_centerlines(source_geometry).set_crs(
    PROJECT_CRS, allow_override=True
)
source_surface = HdfXsec.get_xs_interpolation_surface(source_geometry_hdf).set_crs(
    PROJECT_CRS, allow_override=True
)

model_summary = pd.DataFrame({
    "study": ["Lower Colorado–Cummins eBFE"],
    "huc8": ["12090301"],
    "river_reach": [f"{RIVER} / {REACH}"],
    "cross_sections": [len(source_xs)],
    "footprint_area_mi2": [model_extent.area.iloc[0] / 5280**2],
    "project_crs": [PROJECT_CRS],
    "source_compute": ["successful"],
})
display(model_summary)


## 2. Query real NWM v3 flowlines and classify them by model coverage

Only a small envelope around the model is requested from NOAA. The response is
written to GeoParquet, projected into the model CRS, and passed to
`RasNetworkConflation.classify_edges()`. The output cardinality is one row per
model–edge intersection—not one “winning” edge per RAS reach.


In [ ]:
NWM_SERVICE = (
    "https://maps.water.noaa.gov/server/rest/services/"
    "reference/static_nwm_flowlines/FeatureServer/0/query"
)
wgs_bounds = model_extent.to_crs("EPSG:4326").total_bounds
pad = 0.02
bbox = (
    wgs_bounds[0] - pad,
    wgs_bounds[1] - pad,
    wgs_bounds[2] + pad,
    wgs_bounds[3] + pad,
)
query = {
    "where": "1=1",
    "geometry": ",".join(f"{value:.8f}" for value in bbox),
    "geometryType": "esriGeometryEnvelope",
    "inSR": 4326,
    "spatialRel": "esriSpatialRelIntersects",
    "outFields": "feature_id,name,strm_order,huc6,nwm_vers",
    "returnGeometry": "true",
    "outSR": 4326,
    "f": "geojson",
}
response = requests.get(NWM_SERVICE, params=query, timeout=60)
response.raise_for_status()
nwm_flowlines_wgs84 = gpd.read_file(io.BytesIO(response.content))
nwm_flowlines_wgs84["feature_id"] = nwm_flowlines_wgs84["feature_id"].astype(str)
nwm_cache = RUN_ROOT / "nwm_v3_flowlines.parquet"
nwm_flowlines_wgs84.to_parquet(nwm_cache, index=False)
nwm_edges = nwm_flowlines_wgs84.to_crs(PROJECT_CRS)

coverage_result = RasNetworkConflation.classify_edges(
    model_footprints=model_extent[["geometry_id", "geometry"]],
    network_edges=nwm_edges,
    adapter="nwm",
)
coverage = coverage_result.coverage_df.merge(
    nwm_edges[["feature_id", "name", "nwm_vers"]].drop_duplicates("feature_id"),
    left_on="edge_id",
    right_on="feature_id",
    how="left",
).drop(columns="feature_id")
coverage = gpd.GeoDataFrame(coverage, geometry="geometry", crs=PROJECT_CRS)
coverage["inside_miles"] = coverage["inside_length"] / 5280.0
coverage["edge_miles"] = coverage["edge_length"] / 5280.0
coverage["xs_intersections"] = coverage.geometry.map(
    lambda edge: int(source_xs.intersects(edge).sum())
)

display(
    coverage[[
        "edge_id", "name", "nwm_vers", "extent_status", "inside_fraction",
        "inside_miles", "edge_miles", "xs_intersections",
    ]].style.format({
        "inside_fraction": "{:.1%}",
        "inside_miles": "{:.2f}",
        "edge_miles": "{:.2f}",
    })
)
print(f"Cached modern vector subset: {nwm_cache}")

target_edge = coverage.loc[coverage["edge_id"] == TARGET_EDGE_ID].iloc[0]
assert target_edge["name"] == "Shiloh Branch"
assert target_edge["extent_status"] == "inside"


In [ ]:
extent_color = "#E8DFC8"
ras_color = "#1F2937"
xs_color = "#3B82F6"
partial_color = "#E69F00"
inside_color = "#009E73"
target_color = "#CC0066"

fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
model_extent.plot(ax=ax, color=extent_color, edgecolor="#8B7355", linewidth=1.4, alpha=0.72)
nwm_edges.plot(ax=ax, color="#B8BEC7", linewidth=1.0, alpha=0.65)
coverage.loc[coverage["extent_status"] == "partial"].plot(
    ax=ax, color=partial_color, linewidth=3.2, label="Partial NWM edge"
)
coverage.loc[coverage["extent_status"] == "inside"].plot(
    ax=ax, color=inside_color, linewidth=3.2, label="Fully inside NWM edge"
)
coverage.loc[coverage["edge_id"] == TARGET_EDGE_ID].plot(
    ax=ax, color=target_color, linewidth=6.0, label=f"Target NWM {TARGET_EDGE_ID}"
)
source_centerline.plot(ax=ax, color=ras_color, linewidth=1.3, linestyle="--")
source_xs.plot(ax=ax, color=xs_color, linewidth=0.55, alpha=0.55)

for row in coverage.itertuples(index=False):
    point = row.geometry.interpolate(0.5, normalized=True)
    ax.annotate(
        f"{row.edge_id}\n{row.inside_fraction:.0%} inside",
        (point.x, point.y),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
        color=target_color if row.edge_id == TARGET_EDGE_ID else ras_color,
        fontweight="bold" if row.edge_id == TARGET_EDGE_ID else "normal",
    )

map_bounds = model_extent.total_bounds
map_pad = 0.16 * max(
    map_bounds[2] - map_bounds[0], map_bounds[3] - map_bounds[1]
)
ax.set_xlim(map_bounds[0] - map_pad, map_bounds[2] + map_pad)
ax.set_ylim(map_bounds[1] - map_pad, map_bounds[3] + map_pad)
ax.legend(
    handles=[
        Patch(facecolor=extent_color, edgecolor="#8B7355", label="RAS model footprint"),
        Line2D([0], [0], color=ras_color, linestyle="--", label="RAS river centerline"),
        Line2D([0], [0], color=xs_color, label="RAS cross sections"),
        Line2D([0], [0], color=partial_color, linewidth=3, label="Partial NWM edge"),
        Line2D([0], [0], color=inside_color, linewidth=3, label="Fully inside NWM edge"),
        Line2D([0], [0], color=target_color, linewidth=5, label=f"Target NWM {TARGET_EDGE_ID}"),
    ],
    loc="best",
    fontsize=8,
)
ax.set_title("Real Texas eBFE footprint and intersecting NOAA NWM v3 reaches")
ax.set_aspect("equal")
ax.set_axis_off()
plt.show()


**Verified execution — model extent and NWM coverage.** The source model
footprint contains NWM feature 5790868 completely, while the adjoining upstream
and downstream flowlines are retained as partial-coverage edges. Cross sections
and the HEC-RAS centerline provide the spatial check against the network.

![Texas eBFE model footprint with intersecting NWM reaches](assets/235_1d_breakout_model/01_model_extent_and_nwm_coverage.png)


## 3. Convert NWM edge 5790868 into breakout limits

The edge intersects ten cross sections directly. `select_by_network_edge()`
then includes the next section downstream, matching Ripple1D's shared-boundary
default. For this model the added section is RS 14026, so the independent model
has eleven sections and an internal downstream boundary location.


In [ ]:
direct_selection = RasBreakout1D.select_by_network_edge(
    source_geometry,
    target_edge.geometry,
    river=RIVER,
    reach=REACH,
    downstream_overlap_xs=0,
)
selection = RasBreakout1D.select_by_network_edge(
    source_geometry,
    target_edge.geometry,
    river=RIVER,
    reach=REACH,
)

direct_stations = {float(value) for value in direct_selection.stations}
selected_stations = {float(value) for value in selection.stations}
overlap_stations = selected_stations - direct_stations
assert len(direct_selection.stations) == 10
assert overlap_stations == {14026.0}

source_xs["direct_intersection"] = source_xs["station_num"].isin(direct_stations)
source_xs["selected"] = source_xs["station_num"].isin(selected_stations)
source_xs["downstream_overlap"] = source_xs["station_num"].isin(overlap_stations)
direct_xs = source_xs.loc[source_xs["direct_intersection"]].copy()
overlap_xs = source_xs.loc[source_xs["downstream_overlap"]].copy()
nearby_xs = source_xs.loc[
    source_xs["station_num"].between(
        float(selection.downstream_station) - 1200,
        float(selection.upstream_station) + 1200,
    )
].copy()

display(pd.DataFrame({
    "nwm_edge_id": [TARGET_EDGE_ID],
    "edge_name": [target_edge["name"]],
    "directly_intersected_xs": [len(direct_selection.stations)],
    "downstream_overlap_xs": [", ".join(f"{value:g}" for value in sorted(overlap_stations))],
    "breakout_xs": [len(selection.stations)],
    "upstream_limit": [selection.upstream_station],
    "downstream_limit": [selection.downstream_station],
}))

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), constrained_layout=True)

model_extent.plot(ax=axes[0], color=extent_color, edgecolor="#8B7355", alpha=0.6)
coverage.plot(ax=axes[0], color="#B8BEC7", linewidth=2.0)
coverage.loc[coverage["edge_id"] == TARGET_EDGE_ID].plot(
    ax=axes[0], color=target_color, linewidth=5.0
)
source_xs.plot(ax=axes[0], color=xs_color, linewidth=0.45, alpha=0.5)
axes[0].set_title("A — Target edge within the complete source model")

nearby_xs.plot(ax=axes[1], color="#9CA3AF", linewidth=0.8, alpha=0.6)
direct_xs.plot(ax=axes[1], color=xs_color, linewidth=2.0)
overlap_xs.plot(ax=axes[1], color="#D55E00", linewidth=3.2)
gpd.GeoSeries([target_edge.geometry], crs=PROJECT_CRS).plot(
    ax=axes[1], color=target_color, linewidth=5.0
)
source_centerline.plot(ax=axes[1], color=ras_color, linewidth=1.4, linestyle="--")

for frame, label, color in [
    (direct_xs.nlargest(1, "station_num"), f"US limit RS {selection.upstream_station}", inside_color),
    (overlap_xs, f"DS overlap RS {selection.downstream_station}", "#D55E00"),
]:
    point = frame.geometry.iloc[0].interpolate(0.5, normalized=True)
    axes[1].annotate(
        label, (point.x, point.y), xytext=(7, 7), textcoords="offset points",
        fontsize=9, color=color, fontweight="bold",
    )

xmin, ymin, xmax, ymax = nearby_xs.total_bounds
pad_xy = 0.08 * max(xmax - xmin, ymax - ymin)
axes[1].set_xlim(xmin - pad_xy, xmax + pad_xy)
axes[1].set_ylim(ymin - pad_xy, ymax + pad_xy)
axes[1].set_title("B — Direct intersections plus one downstream overlap XS")
axes[1].legend(
    handles=[
        Line2D([0], [0], color=target_color, linewidth=5, label=f"NWM {TARGET_EDGE_ID}"),
        Line2D([0], [0], color=xs_color, linewidth=2, label="Directly intersected XS"),
        Line2D([0], [0], color="#D55E00", linewidth=3, label="Downstream overlap XS"),
        Line2D([0], [0], color="#9CA3AF", label="Nearby source XS"),
    ],
    loc="best", fontsize=8,
)
for ax in axes:
    ax.set_aspect("equal")
    ax.set_axis_off()
fig.suptitle("How a single NWM reach becomes 1D breakout limits", fontsize=14)
plt.show()


**Verified execution — breakout limits.** Ten blue cross sections intersect
the target NWM edge directly. The orange RS 14026 section is the explicit
downstream overlap used to form an internal boundary for the independent model.

![Directly intersected cross sections and downstream overlap](assets/235_1d_breakout_model/02_nwm_edge_breakout_limits.png)


## 4. Write, validate, and run the independent breakout

Selection and writing remain separate. The extractor copies complete retained
cross-section blocks and steady-flow relationships, creates its own project,
derives an internal downstream boundary from source results, and validates the
destination before execution.


In [ ]:
breakout = RasBreakout1D.extract_selection(
    source_ras,
    RUN_ROOT / f"breakout_nwm_{TARGET_EDGE_ID}",
    selection,
    plan_number=PLAN_NUMBER,
    destination_name=f"Shiloh_NWM_{TARGET_EDGE_ID}",
    source_plan_hdf=source_plan_hdf,
    boundary_mode="auto",
)

display(breakout.validation.checks_df)
assert breakout.validation.is_valid
print(f"Boundary provenance: {breakout.boundary_provenance}")
print(f"Independent project: {breakout.project_file}")

geometry_comparison = RasBreakout1D.compare_geometry(
    source_geometry,
    breakout.geometry_file,
    breakout.selection,
)
display(geometry_comparison)
assert geometry_comparison["content_equal"].all()
assert geometry_comparison.attrs["structure_blocks_equal"] is True

breakout_compute = RasBreakout1D.run(
    breakout,
    verify=True,
    force_rerun=True,
    num_cores=1,
)
assert breakout_compute, "Breakout steady plan did not complete successfully"
breakout_plan_hdf = Path(f"{breakout.plan_file}.hdf")
assert breakout_plan_hdf.is_file()


In [ ]:
breakout_geometry_hdf = Path(f"{breakout.geometry_file}.hdf")
breakout_xs = GeomParser.get_xs_cut_lines(breakout.geometry_file).set_crs(
    PROJECT_CRS, allow_override=True
)
breakout_centerline = GeomParser.get_river_centerlines(breakout.geometry_file).set_crs(
    PROJECT_CRS, allow_override=True
)
breakout_surface = HdfXsec.get_xs_interpolation_surface(breakout_geometry_hdf).set_crs(
    PROJECT_CRS, allow_override=True
)
selected_source_xs = source_xs.loc[source_xs["selected"]].copy()
excluded_source_xs = source_xs.loc[~source_xs["selected"]].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), constrained_layout=True)
source_surface.plot(ax=axes[0], color="#E5E7EB", edgecolor="none", alpha=0.72)
excluded_source_xs.plot(ax=axes[0], color="#9CA3AF", linewidth=0.4, alpha=0.45)
selected_source_xs.plot(ax=axes[0], color=xs_color, linewidth=1.8)
overlap_xs.plot(ax=axes[0], color="#D55E00", linewidth=3.0)
gpd.GeoSeries([target_edge.geometry], crs=PROJECT_CRS).plot(
    ax=axes[0], color=target_color, linewidth=4.5
)
axes[0].set_title(f"Source: 40 XS; selected {len(selected_source_xs)} for NWM {TARGET_EDGE_ID}")

breakout_surface.plot(
    ax=axes[1], color="#CDECCF", edgecolor=inside_color, linewidth=0.8, alpha=0.7
)
breakout_xs.plot(ax=axes[1], color=xs_color, linewidth=1.8)
breakout_centerline.plot(ax=axes[1], color=ras_color, linewidth=1.7, linestyle="--")
axes[1].set_title(f"Breakout: independent project with {len(breakout_xs)} XS")

focus_bounds = selected_source_xs.total_bounds
focus_pad = 0.08 * max(
    focus_bounds[2] - focus_bounds[0], focus_bounds[3] - focus_bounds[1]
)
for ax in axes:
    ax.set_xlim(focus_bounds[0] - focus_pad, focus_bounds[2] + focus_pad)
    ax.set_ylim(focus_bounds[1] - focus_pad, focus_bounds[3] + focus_pad)
    ax.set_aspect("equal")
    ax.set_axis_off()
axes[1].legend(
    handles=[
        Patch(facecolor="#CDECCF", edgecolor=inside_color, label="Breakout interpolation surface"),
        Line2D([0], [0], color=xs_color, linewidth=2, label="Retained XS"),
        Line2D([0], [0], color=ras_color, linestyle="--", label="Retained centerline"),
    ],
    loc="best", fontsize=8,
)
fig.suptitle("Spatial audit: source selection versus written breakout geometry", fontsize=14)
plt.show()


**Verified execution — written geometry.** The left panel identifies the 11
retained sections within the 40-section source model. The right panel reads the
new geometry HDF back from the standalone breakout and plots its cross sections,
centerline, and interpolation surface.

![Source selection compared with written breakout geometry](assets/235_1d_breakout_model/03_source_vs_breakout_geometry.png)


## 5. Compare retained hydraulic results

The comparison is keyed by river, reach, cross section, and profile. The figure
uses the largest source flow profile and shows both the longitudinal water
surface and the source-minus-breakout residual at every retained section.


In [ ]:
results_comparison = RasBreakout1D.compare_results(
    source_plan_hdf,
    breakout_plan_hdf,
    breakout.selection,
)
assert results_comparison["_merge"].eq("both").all()

delta_columns = [
    column for column in results_comparison.columns
    if column.endswith("_delta") and column != "channel_length_delta"
]
delta_summary = (
    results_comparison[delta_columns]
    .abs()
    .agg(["count", "mean", "max"])
    .T
    .sort_index()
)
display(delta_summary)

profile_flows = results_comparison.groupby("profile", observed=True)["flow_source"].max()
comparison_profile = profile_flows.idxmax()
profile_plot = results_comparison.loc[
    results_comparison["profile"] == comparison_profile
].copy()
profile_plot["station_num"] = pd.to_numeric(profile_plot["node_id"], errors="coerce")
profile_plot = profile_plot.sort_values("station_num", ascending=False)

source_profile_xs = HdfXsec.get_cross_sections(source_geometry_hdf)
source_profile_xs = source_profile_xs.loc[
    (source_profile_xs["River"] == RIVER) & (source_profile_xs["Reach"] == REACH)
].copy()
source_profile_xs["station_num"] = pd.to_numeric(source_profile_xs["RS"], errors="coerce")
source_profile_xs["channel_invert"] = source_profile_xs["station_elevation"].apply(
    lambda points: min(float(point[1]) for point in points)
)
profile_plot = profile_plot.merge(
    source_profile_xs[["station_num", "channel_invert"]],
    on="station_num",
    how="left",
)

fig, axes = plt.subplots(
    2, 1, figsize=(13, 7.5), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True,
)
axes[0].plot(
    profile_plot["station_num"], profile_plot["channel_invert"],
    color=ras_color, linewidth=1.7, marker=".", label="Source channel invert",
)
axes[0].fill_between(
    profile_plot["station_num"], profile_plot["channel_invert"],
    profile_plot["channel_invert"].min() - 2.0,
    color="#E5E7EB", alpha=0.8,
)
axes[0].plot(
    profile_plot["station_num"], profile_plot["wsel_source"],
    color=xs_color, linewidth=2.4, marker="o", markersize=4, label="Source WSE",
)
axes[0].plot(
    profile_plot["station_num"], profile_plot["wsel_destination"],
    color=target_color, linewidth=1.8, linestyle="--", marker="s", markersize=3.5,
    label="Breakout WSE",
)
axes[0].set_ylabel("Elevation (ft)")
axes[0].set_title(
    f"{comparison_profile} — {profile_flows.loc[comparison_profile]:,.0f} cfs"
)
axes[0].ticklabel_format(style="plain", axis="y", useOffset=False)
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(loc="best")

wse_delta_inches = profile_plot["wsel_delta"] * 12.0
axes[1].axhline(0.0, color=ras_color, linewidth=1.0)
axes[1].plot(
    profile_plot["station_num"], wse_delta_inches,
    color=inside_color, linewidth=1.8, marker="o", markersize=4,
)
axes[1].fill_between(
    profile_plot["station_num"], 0.0, wse_delta_inches,
    color=inside_color, alpha=0.18,
)
axes[1].set_ylabel("WSE delta (in)")
axes[1].set_xlabel("River station (ft; flow direction →)")
axes[1].grid(axis="y", alpha=0.25)
axes[1].invert_xaxis()
max_delta_inches = float(wse_delta_inches.abs().max())
delta_limit = max(max_delta_inches * 1.25, 0.001)
axes[1].set_ylim(-delta_limit, delta_limit)
axes[1].text(
    0.01, 0.93, f"Maximum |WSE delta| = {max_delta_inches:.6f} in",
    transform=axes[1].transAxes, va="top", fontsize=9,
)
fig.suptitle("Source and NWM-reach breakout hydraulic comparison", fontsize=14)
plt.show()

display(results_comparison[[
    "river", "reach", "node_id", "profile",
    "flow_source", "flow_destination", "flow_delta",
    "wsel_source", "wsel_destination", "wsel_delta", "_merge",
]].head(18))


**Verified execution — hydraulic agreement.** The source and breakout water
surfaces overlay for the highest-flow profile. The lower panel exposes the
source-minus-breakout residual in inches rather than hiding it beneath the WSE
lines; all 77 retained cross-section/profile records were present in both runs.

![Source and breakout water surfaces with residuals](assets/235_1d_breakout_model/04_hydraulic_results_comparison.png)


## Result

The destination is a standalone 1D steady HEC-RAS project for NWM feature
5790868. It retains the ten sections directly intersected by that NWM reach,
adds one downstream overlap section, preserves the complete selected geometry
and steady-flow content, passes validation, runs successfully, and exposes a
section-by-section hydraulic comparison with the source eBFE model.

This is the intended relationship between conflation and trimming:
**model extent finds all in-domain network edges; one chosen edge defines one
auditable `RasBreakout1D`.**
